<a href="https://colab.research.google.com/github/pratikshabole/generate_and_critique_essay/blob/main/essay.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install duckduckgo-search langchain-community -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [3]:
!pip install -U ddgs -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 2.4 MB/s eta 0:00:00


In [4]:
!pip install langchain langchain-groq langchain-core -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 1.3 MB/s eta 0:00:00


In [5]:
!pip install streamlit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 40.0 MB/s eta 0:00:00


In [6]:
!pip install pyngrok -q

In [7]:
from pyngrok import ngrok

In [17]:
!pip install -q langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 2.2 MB/s eta 0:00:00


In [24]:
%%writefile essayapp.py

import streamlit as st
import os
from typing import Dict, Any, TypedDict
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_groq import ChatGroq

# Set up page styling
st.set_page_config(page_title="✍️ Agentic Essay Writer", layout="wide")
st.title("✍️ Agentic Essay Writer & Self-Critique Agent")
st.write("This agent generates an initial draft, critiques its own work, and refines it dynamically.")

# ---------------------------------------------------------------------
# 1. State & Agent Definition
# ---------------------------------------------------------------------
class AgentState(TypedDict):
    topic: str
    draft: str
    critique: str
    revision_count: int
    final_essay: str

def get_llm():
    # Pulls API key from environment
    return ChatGroq(temperature=0, model_name='llama-3.3-70b-versatile')

def generate_initial_draft(state: AgentState) -> Dict[str, Any]:
    llm = get_llm()
    messages = [
        SystemMessage(content="You are an expert academic essayist. Write a concise, compelling 3-paragraph essay on the given topic."),
        HumanMessage(content=f"Write an essay about: {state['topic']}")
    ]
    response = llm.invoke(messages)
    return {"draft": response.content, "revision_count": 1}

def critique_draft(state: AgentState) -> Dict[str, Any]:
    llm = get_llm()
    messages = [
        SystemMessage(content=(
            "You are a strict, constructive writing editor. Review the essay draft. "
            "Identify structural gaps, clarity improvements, or logical flaws. "
            "If the essay is exceptional and needs no changes, write 'APPROVED'."
        )),
        HumanMessage(content=f"Review this essay draft:\n\n{state['draft']}")
    ]
    response = llm.invoke(messages)
    return {"critique": response.content}

def revise_draft(state: AgentState) -> Dict[str, Any]:
    llm = get_llm()
    messages = [
        SystemMessage(content="You are an adaptive writer. Rewrite the essay draft incorporating all the constructive critique points provided."),
        HumanMessage(content=f"Original Draft:\n{state['draft']}\n\nCritique:\n{state['critique']}")
    ]
    response = llm.invoke(messages)
    return {
        "draft": response.content,
        "revision_count": state["revision_count"] + 1
    }

# ---------------------------------------------------------------------
# 2. Control Loop Execution
# ---------------------------------------------------------------------
def run_essay_agent(topic: str, max_iterations: int = 3):
    # Initialize state
    state: AgentState = {
        "topic": topic,
        "draft": "",
        "critique": "",
        "revision_count": 0,
        "final_essay": ""
    }

    status_box = st.status("🚀 Initializing Agent Loop...", expanded=True)

    # Run loop
    while state["revision_count"] < max_iterations:
        current_rev = state["revision_count"] + 1

        status_box.write(f"📝 Generating / Modifying Draft (Pass {current_rev})...")
        if state["revision_count"] == 0:
            draft_res = generate_initial_draft(state)
        else:
            draft_res = revise_draft(state)
        state.update(draft_res)

        status_box.write(f"🔍 Executing self-critique protocol on Draft {current_rev}...")
        critique_res = critique_draft(state)
        state.update(critique_res)

        # Break out early if approved
        if "APPROVED" in state["critique"].upper():
            status_box.write("✅ Critique approved the text structural qualities!")
            break

    state["final_essay"] = state["draft"]
    status_box.update(label="✨ Agent Pipeline Complete!", state="complete")
    return state

# ---------------------------------------------------------------------
# 3. Streamlit Interface UI Layout
# ---------------------------------------------------------------------
# Sidebar configuration
with st.sidebar:
    st.header("🔑 Authentication & Control")
    api_key = st.text_input("Grok API Key", type="password")
    max_loops = st.slider("Max Revision Cycles", min_value=1, max_value=4, value=2)

# Main container
topic_input = st.text_input("Enter your essay topic or prompt:", value="The socio-economic impacts of space exploration")

if st.button("Generate & Critique Essay"):
    if not api_key:
        st.error("Please provide your Groq API key in the sidebar to proceed.")
    elif not topic_input.strip():
        st.warning("Please input a valid topic.")
    else:
        # Save key to environment variable securely inside execution context
        os.environ["GROQ_API_KEY"] = api_key

        # Run agent loop execution
        final_state = run_essay_agent(topic_input, max_iterations=max_loops)

        # Separate output displays elegantly inside layouts
        col1, col2 = st.columns(2)

        with col1:
            st.subheader("📚 Polished Final Essay")
            st.info(f"Total Revision Cycles Completed: {final_state['revision_count']}")
            st.write(final_state["final_essay"])

        with col2:
            st.subheader("📋 Last Critique Feedback Report")
            st.markdown(f"```\n{final_state['critique']}\n```")

Writing essayapp.py


In [17]:
from pyngrok import ngrok
# Kill previous lingering tunnels
ngrok.kill()

In [21]:
import os

# Set your active ngrok authtoken if required
# !ngrok config add-authtoken YOUR_NGROK_TOKEN

# Spin up the background Streamlit process
import subprocess
ngrok.set_auth_token("3Aqm01Mhfgr8QnsWNpFBJ8fuoaO_7xqPGuUJS4BiC47vW8cAX")

subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])

# Direct traffic securely out via ngrok on port 8501
public_url = ngrok.connect(8501)
print("🚀 Your Critiquing Agent Dashboard is live at:", public_url)

from pyngrok import ngrok
import subprocess
import time



🚀 Your Critiquing Agent Dashboard is live at: NgrokTunnel: "https://hedgeless-figurally-mayson.ngrok-free.dev" -> "http://localhost:8501"
